In [ ]:
%pip install confluent-kafka

In [ ]:

import uuid
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType

In [ ]:
dbutils.widgets.dropdown(
    name="stage",
    defaultValue="CCD",
    choices=["CCD", "CDAM", "CASE_LINKING"],
    label="Stage"
)
stage = dbutils.widgets.get("stage")

In [ ]:
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key   = config.first()["lz_key"].strip().lower()
print(f"env_name={env_name}, lz_key={lz_key}")

keyvault_name = f"ingest{lz_key}-meta002-{env_name}"
print(f"keyvault_name={keyvault_name}")

In [ ]:
try:
    client_secret = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-SECRET')
    tenant_id     = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-TENANT-ID')
    client_id     = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-ID')
    print("Successfully retrieved all Service Principal secrets from Key Vault")
except Exception as e:
    print(f"Failed to retrieve secrets: {e}", exc_info=True)
    raise

In [ ]:
curated_storage_account = f"ingest{lz_key}curated{env_name}"
try:
    configs = {
        f"fs.azure.account.auth.type.{curated_storage_account}.dfs.core.windows.net": "OAuth",
        f"fs.azure.account.oauth.provider.type.{curated_storage_account}.dfs.core.windows.net":
            "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
        f"fs.azure.account.oauth2.client.id.{curated_storage_account}.dfs.core.windows.net": client_id,
        f"fs.azure.account.oauth2.client.secret.{curated_storage_account}.dfs.core.windows.net": client_secret,
        f"fs.azure.account.oauth2.client.endpoint.{curated_storage_account}.dfs.core.windows.net":
            f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
    }
    for key, val in configs.items():
        spark.conf.set(key, val)
    print(f"Configured OAuth for {curated_storage_account}")
except Exception as e:
    print(f"Failed to configure OAuth: {e}", exc_info=True)
    raise

In [ ]:
silver_base = f"abfss://silver@{curated_storage_account}.dfs.core.windows.net"

STAGE_CONFIG = {
    "CCD": {
        "pub_audit_path": f"{silver_base}/ARIADM/ACTIVE/CCD/APPEALS/publish_payload_audit",
        "ack_audit_path": f"{silver_base}/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/all_active_states/ack_audit",
        "key":            "CaseNo",
        "topic":          f"evh-active-pub-{env_name}-{lz_key}-uks-dlrm-01",
    },
    "CDAM": {
        "pub_audit_path": f"{silver_base}/ARIADM/ACTIVE/CCD/APPEALS/CDAM/publish_audit_db_eh",
        "ack_audit_path": f"{silver_base}/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/CDAM/ack_audit",
        "key":            "CaseNo",
        "topic":          f"evh-active-cdam-pub-{env_name}-{lz_key}-uks-dlrm-01",
    },
    "CASE_LINKING": {
        "pub_audit_path": f"{silver_base}/ARIADM/ACTIVE/CCD/APPEALS/CASE_LINKING/publish_audit_db_eh",
        "ack_audit_path": f"{silver_base}/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/CASE_LINKING/ack_audit",
        "key":            "CCDCaseReferenceNumber",
        "topic":          f"evh-active-caselink-pub-{env_name}-{lz_key}-uks-dlrm-01",
    },
}

if stage not in STAGE_CONFIG:
    raise ValueError(f"Unknown stage '{stage}'. Valid options: {list(STAGE_CONFIG)}")

cfg = STAGE_CONFIG[stage]
print(f"pub_audit_path : {cfg['pub_audit_path']}")
print(f"ack_audit_path : {cfg['ack_audit_path']}")
print(f"topic          : {cfg['topic']}")

## Audit Comparison

In [ ]:
pub_df = spark.read.format("delta").load(cfg["pub_audit_path"])
published = (
    pub_df
    .filter(col("Status") == "SUCCESS")
    .select(col(cfg["key"]))
    .distinct()
)
published_count = published.count()
print(f"Successfully published: {published_count}")

In [ ]:
ack_df = spark.read.format("delta").load(cfg["ack_audit_path"])
acknowledged = (
    ack_df
    .filter(col("Status") == "SUCCESS")
    .select(col(cfg["key"]))
    .distinct()
)
acknowledged_count = acknowledged.count()
print(f"Successfully acknowledged: {acknowledged_count}")

In [ ]:
missing_keys = published.join(acknowledged, on=cfg["key"], how="left_anti").cache()
missing_count = missing_keys.count()
print(f"Published but not acknowledged: {missing_count}")

In [ ]:
missing_keys.display()

In [ ]:
if missing_count == 0:
    print("No missing events to republish. Exiting.")
    dbutils.notebook.exit("Nothing to republish — all published records have been acknowledged")

## Republish Missing Events

In [ ]:
eh_kv_secret = dbutils.secrets.get(scope=keyvault_name, key="RootManageSharedAccessKey")
eventhubs_hostname = f"ingest{lz_key}-integration-eventHubNamespace001-{env_name}.servicebus.windows.net:9093"

base_kafka_conf = {
    'bootstrap.servers': eventhubs_hostname,
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': '$ConnectionString',
    'sasl.password': eh_kv_secret,
}

producer_conf = {
    **base_kafka_conf,
    'retries': 5,
    'enable.idempotence': True,
}

consumer_conf = {
    **base_kafka_conf,
    'group.id': f'republish-{stage}-{uuid.uuid4()}',
    'enable.auto.commit': False,
    'auto.offset.reset': 'earliest',
}

broadcast_producer_conf = sc.broadcast(producer_conf)
broadcast_consumer_conf = sc.broadcast(consumer_conf)

In [ ]:
from confluent_kafka import Consumer, TopicPartition
from datetime import datetime, timezone

result_schema_map = {
    "CCD": StructType([
        StructField("RunID",              StringType(), True),
        StructField("AriaCaseNo",         StringType(), True),
        StructField("CaseNo",             StringType(), True),
        StructField("Filename",           StringType(), True),
        StructField("State",              StringType(), True),
        StructField("PublishingDateTime", StringType(), True),
        StructField("Status",             StringType(), True),
        StructField("PublishContent",     StringType(), True),
        StructField("Error",              StringType(), True),
    ]),
    "CDAM": StructType([
        StructField("RunID",              StringType(), True),
        StructField("CaseNo",             StringType(), True),
        StructField("FileName",           StringType(), True),
        StructField("FileURL",            StringType(), True),
        StructField("PublishingDateTime", StringType(), True),
        StructField("Status",             StringType(), True),
        StructField("Error",              StringType(), True),
    ]),
    "CASE_LINKING": StructType([
        StructField("RunID",                  StringType(), True),
        StructField("CCDCaseReferenceNumber", StringType(), True),
        StructField("CaseLinkCount",          StringType(), True),
        StructField("PublishingDateTime",     StringType(), True),
        StructField("Status",                 StringType(), True),
        StructField("Error",                  StringType(), True),
    ]),
}
result_schema = result_schema_map[stage]


def build_result_row(stage_name, rd, ts, status, error, key_val):
    if stage_name == "CCD":
        return (
            rd.get("RunID", ""), rd.get("AriaCaseNo", ""),
            rd.get("CaseNo", key_val), rd.get("CaseNo", key_val),
            rd.get("State", ""), ts, status,
            rd.get("Content", "") if status == "SUCCESS" else "",
            error,
        )
    elif stage_name == "CDAM":
        return (
            rd.get("RunID", ""), rd.get("CaseNo", key_val),
            rd.get("FileName", ""), rd.get("FileURL", ""),
            ts, status, error,
        )
    elif stage_name == "CASE_LINKING":
        case_links = (rd.get("CaseLinkPayload") or {}).get("caseLinks") or []
        return (
            rd.get("RunID", ""), rd.get("CCDCaseReferenceNumber", key_val),
            str(len(case_links)), ts, status, error,
        )
    else:
        raise RuntimeError("Unable to match stage")


# Keys of records that were published but never acknowledged - these are what
# we need to find in the EventHub itself (matched on partition key) and replay.
missing_key_list = [row[cfg["key"]] for row in missing_keys.collect()]
missing_key_set  = set(missing_key_list)
broadcast_keys   = sc.broadcast(missing_key_set)
print(f"Keys to locate in EventHub and republish: {len(missing_key_set)}")

# Discover partitions and their current offset range so each Spark task can
# scan a bounded slice of the topic in parallel.
probe_consumer = Consumer({**consumer_conf, 'group.id': f'republish-probe-{uuid.uuid4()}'})
topic_metadata = probe_consumer.list_topics(cfg["topic"], timeout=30).topics[cfg["topic"]]
partition_ranges = []
for partition_id in topic_metadata.partitions:
    tp = TopicPartition(cfg["topic"], partition_id)
    low, high = probe_consumer.get_watermark_offsets(tp, timeout=30, cached=False)
    if high > low:
        partition_ranges.append((partition_id, low, high))
probe_consumer.close()
print(f"Partitions to scan: {partition_ranges}")


def scan_and_republish(partitions_iter):
    import json
    from confluent_kafka import Consumer, Producer, TopicPartition
    from datetime import datetime, timezone

    keys_to_find = broadcast_keys.value
    results = []
    if not keys_to_find:
        return results

    consumer = Consumer(broadcast_consumer_conf.value)
    producer = Producer(**broadcast_producer_conf.value)

    def make_callback(rd, key_str):
        def callback(err, _msg):
            ts     = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")
            status = "ERROR" if err else "SUCCESS"
            error  = str(err) if err else ""
            results.append(build_result_row(stage, rd, ts, status, error, key_str))
        return callback

    for partition_id, low, high in partitions_iter:
        remaining = set(keys_to_find)
        tp = TopicPartition(cfg["topic"], partition_id, low)
        consumer.assign([tp])

        while remaining and consumer.position([tp])[0].offset < high:
            msg = consumer.poll(timeout=10)
            if msg is None:
                break
            if msg.error():
                continue

            raw_key = msg.key()
            key_str = raw_key.decode("utf-8") if raw_key else None
            if key_str not in remaining:
                continue

            value_bytes = msg.value()
            try:
                rd = json.loads(value_bytes.decode("utf-8"))
            except Exception:
                rd = {}

            cb = make_callback(rd, key_str)
            try:
                producer.produce(topic=cfg["topic"], key=raw_key, value=value_bytes, callback=cb)
            except BufferError:
                producer.poll(10)
                producer.produce(topic=cfg["topic"], key=raw_key, value=value_bytes, callback=cb)
            producer.poll(0)
            remaining.discard(key_str)

    try:
        producer.flush()
    except Exception as e:
        print(f"Flush error: {e}")
    consumer.close()
    return results


if partition_ranges:
    result_rdd = sc.parallelize(partition_ranges, numSlices=len(partition_ranges)).mapPartitions(scan_and_republish)
    result_df  = spark.createDataFrame(result_rdd, result_schema).cache()
else:
    result_df = spark.createDataFrame([], result_schema)

# Any keys not found in the topic (eg. outside EventHub retention) are recorded
# as failures so the audit trail still accounts for every missing key.
found_keys      = {row[cfg["key"]] for row in result_df.select(cfg["key"]).distinct().collect()}
unresolved_keys = missing_key_set - found_keys
if unresolved_keys:
    print(f"Warning: {len(unresolved_keys)} key(s) not found in EventHub (may be outside retention): {sorted(unresolved_keys)}")
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")
    unresolved_rows = [build_result_row(stage, {}, ts, "ERROR", "Key not found in EventHub retention window", k) for k in unresolved_keys]
    result_df = result_df.unionByName(spark.createDataFrame(unresolved_rows, result_schema)).cache()

result_df.display()

In [ ]:
failed_df = result_df.filter(col("Status") == "ERROR")
failed_df.display()
print(f"Failed: {failed_df.count()}")

In [ ]:
successful = result_df.filter(col("Status") == "SUCCESS").count()
failed     = result_df.filter(col("Status") == "ERROR").count()

print(f"Stage            : {stage}")
print(f"Missing events   : {missing_count}")
print(f"Republish success: {successful}")
print(f"Republish failed : {failed}")

dbutils.notebook.exit({"stage": stage, "missing": missing_count, "republish_success": successful, "republish_failed": failed})